# DL

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# Optuna 탐색은 train 파트만 사용해도 충분하며, 길이 불일치 문제를 방지합니다.
X_opt_source = X_train_part.reset_index(drop=True)
y_opt_source = y_train_part.reset_index(drop=True)

print(f"🔎 Optuna split 입력 길이 - X_opt_source: {len(X_opt_source)}, y_opt_source: {len(y_opt_source)}")
if len(X_opt_source) != len(y_opt_source):
    raise ValueError(f"Optuna용 X와 y 길이가 다릅니다. X={len(X_opt_source)}, y={len(y_opt_source)}")

X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(
    X_opt_source, y_opt_source, test_size=0.2, random_state=42, stratify=y_opt_source
)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        # 은닉층(Hidden Layer)의 크기와 개수를 컴퓨터가 조합해 봅니다.
        'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(64, 32), (128, 64, 32), (64,)]),
        # 과적합 방지를 위한 L2 규제 (XGB의 reg_lambda와 비슷)
        'alpha': trial.suggest_float('alpha', 0.0001, 0.1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 0.001, 0.05, log=True),
        'max_iter': 1000,          # 충분한 반복 횟수
        'early_stopping': True,    # 신경망 자체적인 조기 종료 켜기
        'random_state': 42
    }
    
    model_opt = MLPClassifier(**params)
    
    # 🚨 신경망도 내부적으로 분할하므로 여기서 eval_set은 넣지 않습니다.
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = MLPClassifier(
        **study.best_params,
        max_iter=1000,
        early_stopping=True,
        random_state=42 + fold
    )
    
    # 🚨 eval_set 없음
    model.fit(X_tr, y_tr)
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


📥 데이터를 불러오고 K-Fold를 위해 병합합니다...
✅ 병합 완료! (전체 학습 데이터 X: (200000, 44), y: (200000,))

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...
🔎 Optuna split 입력 길이 - X_opt_source: 160000, y_opt_source: 160000


c:\Users\kym92\.conda\envs\yu-m-n-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-09 17:39:53,590] A new study created in memory with name: no-name-876dab59-c7b9-41ed-9a6b-a4bbaae5404f
c:\Users\kym92\.conda\envs\yu-m-n-env\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (64, 32) which is of type tuple.
  optuna_warn(message)
c:\Users\kym92\.conda\envs\yu-m-n-env\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (128, 64, 32) which is of type tuple.
  optuna_warn(message)
c:\Users\kym92\.conda\envs\yu-m-n-env\Lib\site-packages\op

🎉 탐색 완료! 최고 Validation AUC: 0.5895

🚀 XGBoost K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5890)
   - Fold 2 완료 (ROC-AUC: 0.5881)
   - Fold 3 완료 (ROC-AUC: 0.5947)
   - Fold 4 완료 (ROC-AUC: 0.5805)
   - Fold 5 완료 (ROC-AUC: 0.5861)

--- 📊 최종 Validation(OOF) 결과 ---
정확도 (Accuracy): 0.5652
F1 점수 (F1 Score): 0.4940
AUC (ROC-AUC):  0.5864
Fold별 AUC: [0.589, 0.5881, 0.5947, 0.5805, 0.5861]


## MLP 대시보드용 CSV 추출

In [2]:
import numpy as np
import pandas as pd

model_name = "mlp"

print(f"\n💾 {model_name} 대시보드용 CSV 파일 추출을 시작합니다...")

# Validation 파트 길이만큼 마지막 구간을 잘라서 저장
val_size = len(X_val_part)

val_prob = np.asarray(oof_pred)[-val_size:]
val_label = np.asarray(oof_label)[-val_size:]
val_target = y_val_part.values

# 1. Validation 예측 결과
val_df = pd.DataFrame({
    "pred_prob": val_prob,
    "pred_label": val_label,
    "returned": val_target
})
val_df.to_csv(f"val_predictions_{model_name}.csv", index=False)

# 2. Test 예측 결과
test_label = (np.asarray(test_pred) >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": np.asarray(test_pred),
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_name}.csv", index=False)

print(f"✅ val_predictions_{model_name}.csv 생성 완료! (rows={len(val_df)})")
print(f"✅ test_predictions_{model_name}.csv 생성 완료! (rows={len(test_df)})")
print(f"✅ 검증 행 수 확인: X_val_part={len(X_val_part)}, y_val_part={len(y_val_part)}, 저장된 val_df={len(val_df)}")
print(f"⚠️ {model_name}은 신경망 모델이라 feature_importance 파일은 생성하지 않습니다.")


💾 mlp 대시보드용 CSV 파일 추출을 시작합니다...
✅ val_predictions_mlp.csv 생성 완료! (rows=40000)
✅ test_predictions_mlp.csv 생성 완료! (rows=50000)
✅ 검증 행 수 확인: X_val_part=40000, y_val_part=40000, 저장된 val_df=40000
⚠️ mlp은 신경망 모델이라 feature_importance 파일은 생성하지 않습니다.


## Main

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...")

# 🚨 딥러닝은 반드시 스케일링된 데이터를 사용해야 합니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')
y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

# 테스트 데이터를 미리 텐서(딥러닝 전용 데이터 타입)로 변환해 둡니다.
X_test_tensor = torch.FloatTensor(X_test.values)

# ==========================================
# 🧩 부품 1: 신경망 아키텍처 (도면 설계)
# ==========================================
class ReturnRiskNet(nn.Module):
    def __init__(self, input_dim):
        super(ReturnRiskNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),   # 학습을 안정적으로 만들어주는 부품
            nn.ReLU(),
            nn.Dropout(0.3),      # 과적합 방지
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(32, 1),
            nn.Sigmoid()          # 0~1 사이 확률값 출력
        )

    def forward(self, x):
        return self.network(x)

# ==========================================
# 2. K-Fold 설정 및 PyTorch 학습 루프 시작
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []

# 학습용 파라미터 (원래라면 Optuna로 찾았을 값들을 고정값으로 설정)
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 0.005

print("\n🚀 PyTorch K-Fold 딥러닝 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")
    
    # 1) Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx].values, X.iloc[val_idx].values
    y_tr, y_va = y.iloc[train_idx].values, y.iloc[val_idx].values
    
    # 2) 🧩 부품 2: 데이터 로더 (텐서로 변환하고 식판에 담기)
    # y값의 형태를 (N,) 에서 (N, 1) 로 세워주어야 딥러닝이 인식합니다.
    train_dataset = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).unsqueeze(1))
    val_dataset = TensorDataset(torch.FloatTensor(X_va), torch.FloatTensor(y_va).unsqueeze(1))
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3) 모델, 손실 함수, 최적화 도구 세팅
    model = ReturnRiskNet(input_dim=X.shape[1])
    criterion = nn.BCELoss() # 이진 분류용 손실 함수
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 4) 🧩 부품 3: 학습 루프 (수동 변속 과정)
    for epoch in range(EPOCHS):
        model.train() # 학습 모드 ON
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()            # 1. 기울기 초기화
            predictions = model(batch_X)     # 2. 예측
            loss = criterion(predictions, batch_y) # 3. 오차 계산
            loss.backward()                  # 4. 역전파 (어디 고칠지 계산)
            optimizer.step()                 # 5. 가중치 업데이트
            
    # 5) 검증 데이터 예측 및 OOF 저장
    model.eval() # 평가 모드 ON (Dropout 등 정지)
    with torch.no_grad(): # 예측 시에는 기울기 계산을 끔(메모리 절약)
        val_proba = model(torch.FloatTensor(X_va)).squeeze().numpy()
        oof_pred[val_idx] = val_proba
        
        # 테스트 데이터 예측 (5로 나누어 누적)
        test_proba = model(X_test_tensor).squeeze().numpy()
        test_pred += test_proba / skf.n_splits
        
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
oof_label = (oof_pred >= 0.5).astype(int)
print(f"\n🏆 최종 PyTorch OOF ROC-AUC: {overall_auc:.4f}")



📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...

🚀 PyTorch K-Fold 딥러닝 학습을 시작합니다...

--- Fold 1 ---


C:\Users\kym92\AppData\Local\Temp\ipykernel_25316\1558699533.py:76: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_dataset = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).unsqueeze(1))


Fold 1 완료 (ROC-AUC: 0.5932)

--- Fold 2 ---
Fold 2 완료 (ROC-AUC: 0.5895)

--- Fold 3 ---
Fold 3 완료 (ROC-AUC: 0.5927)

--- Fold 4 ---
Fold 4 완료 (ROC-AUC: 0.5842)

--- Fold 5 ---
Fold 5 완료 (ROC-AUC: 0.5896)

🏆 최종 PyTorch OOF ROC-AUC: 0.5893


In [4]:
# ==========================================
# 4. 대시보드 연동용 CSV 일괄 추출 (PyTorch 기본모델용)
# ==========================================
import numpy as np
import pandas as pd

model_name = "pytorch_base"
print(f"\n💾 {model_name} 대시보드 및 제출용 CSV 파일 추출을 시작합니다...")

# Validation 파트 길이만큼 마지막 구간을 잘라서 저장
val_size = len(X_val_part)

val_prob = np.asarray(oof_pred)[-val_size:]
val_label = np.asarray(oof_label)[-val_size:]
val_target = y_val_part.values

# 1. Validation 예측 결과
val_df = pd.DataFrame({
    "pred_prob": val_prob,
    "pred_label": val_label,
    "returned": val_target
})
val_df.to_csv(f"val_predictions_{model_name}.csv", index=False)

# 2. Test 예측 결과 및 제출용
test_prob = np.asarray(test_pred)
test_label = (test_prob >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": test_prob,
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_name}.csv", index=False)
test_df[["order_id", "pred_prob"]].rename(columns={"pred_prob": "returned"}).to_csv(
    f"{model_name}_submission.csv", index=False
)

print(f"✅ val_predictions_{model_name}.csv 생성 완료! (rows={len(val_df)})")
print(f"✅ test_predictions_{model_name}.csv 생성 완료! (rows={len(test_df)})")
print(f"✅ {model_name}_submission.csv 생성 완료!")
print(f"✅ 검증 행 수 확인: X_val_part={len(X_val_part)}, y_val_part={len(y_val_part)}, 저장된 val_df={len(val_df)}")
print("⚠️ PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 feature_importance 파일은 생성하지 않습니다.")

# 대시보드 에러 방지용 백업
X_val_part.to_csv("X_val_scaled_backup.csv", index=False)
pd.DataFrame({"returned": y_val_part}).to_csv("y_val_backup.csv", index=False)
print("\n🎉 파이토치 기본모델 추출 완료!")


💾 pytorch_base 대시보드 및 제출용 CSV 파일 추출을 시작합니다...
✅ val_predictions_pytorch_base.csv 생성 완료! (rows=40000)
✅ test_predictions_pytorch_base.csv 생성 완료! (rows=50000)
✅ pytorch_base_submission.csv 생성 완료!
✅ 검증 행 수 확인: X_val_part=40000, y_val_part=40000, 저장된 val_df=40000
⚠️ PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 feature_importance 파일은 생성하지 않습니다.

🎉 파이토치 기본모델 추출 완료!


In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import optuna

print("📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...")

# 🚨 딥러닝 전용 데이터 (Scaled)
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')
y_train_part = pd.read_csv('y_tr.csv').iloc[:, 0].reset_index(drop=True)
y_val_part = pd.read_csv('y_val.csv').iloc[:, 0].reset_index(drop=True)

# 전체 학습 데이터(Train+Validation)를 합쳐 최종 학습/OOF 생성에 사용
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

# 길이 이상 시 강제로 재정렬/복구
X = X.reset_index(drop=True)
y = pd.Series(y).reset_index(drop=True)

print(f"✅ 길이 확인 - X: {len(X)}, y: {len(y)}")
if len(X) != len(y):
    raise ValueError(f"X와 y 길이가 다릅니다. X={len(X)}, y={len(y)}")

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

X_test_tensor = torch.FloatTensor(X_test.values)

# ==========================================
# 🧩 부품 1: 유연한 신경망 아키텍처 (Optuna가 조립할 수 있게 설계)
# ==========================================
class DynamicReturnNet(nn.Module):
    def __init__(self, input_dim, h1, h2, drop_rate):
        super(DynamicReturnNet, self).__init__()
        # Optuna가 던져주는 h1(1층 두께), h2(2층 두께), drop_rate(규제)에 따라 모양이 바뀝니다.
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            
            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            
            nn.Linear(h2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# ==========================================
# 🌟 2. Optuna 하이퍼파라미터 탐색 (임시 8:2 분할로 빠르게)
# ==========================================
print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 분할 사용)...")

# Optuna 탐색은 전체 데이터 대신 train 파트만 사용해도 충분하며, 길이 불일치 문제를 방지합니다.
X_opt_source = X_train_part.reset_index(drop=True)
y_opt_source = y_train_part.reset_index(drop=True)

print(f"🔎 Optuna split 입력 길이 - X_opt_source: {len(X_opt_source)}, y_opt_source: {len(y_opt_source)}")

X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(
    X_opt_source, y_opt_source, test_size=0.2, random_state=42, stratify=y_opt_source
)

# 텐서 및 식판(DataLoader) 준비
opt_train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_opt_tr.values), torch.FloatTensor(y_opt_tr.values).unsqueeze(1)),
    batch_size=256, shuffle=True
)
X_opt_val_tensor = torch.FloatTensor(X_opt_val.values)

def objective(trial):
    # 1. 탐색할 파라미터 (층의 두께, 드롭아웃 비율, 학습률)
    h1 = trial.suggest_categorical('h1', [64, 128, 256])
    h2 = trial.suggest_categorical('h2', [16, 32, 64])
    drop_rate = trial.suggest_float('drop_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 0.001, 0.01, log=True)
    
    # 2. 모델 및 학습 도구 조립
    model_opt = DynamicReturnNet(X.shape[1], h1, h2, drop_rate)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model_opt.parameters(), lr=lr)
    
    # 3. 짧게 학습 (Optuna 탐색용이므로 15 에폭만 돌립니다)
    model_opt.train()
    for epoch in range(15):
        for batch_X, batch_y in opt_train_loader:
            optimizer.zero_grad()
            loss = criterion(model_opt(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    # 4. 검증 점수(AUC) 계산
    model_opt.eval()
    with torch.no_grad():
        val_preds = model_opt(X_opt_val_tensor).squeeze().numpy()
        
    return roc_auc_score(y_opt_val, val_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 20번 탐색
print(f"\n🎉 탐색 완료! 최적의 파라미터: {study.best_params}")

# ==========================================
# 3. K-Fold 설정 및 PyTorch 최종 학습 (최적 파라미터 적용)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []

# Optuna가 찾은 최고 설정값 가져오기
best_h1 = study.best_params['h1']
best_h2 = study.best_params['h2']
best_drop = study.best_params['drop_rate']
best_lr = study.best_params['lr']

print("\n🚀 Optuna 최적 파라미터로 PyTorch K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx].values, X.iloc[val_idx].values
    y_tr, y_va = y.iloc[train_idx].values, y.iloc[val_idx].values
    
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).unsqueeze(1)),
        batch_size=256, shuffle=True
    )
    
    # Optuna가 찾은 도면대로 신경망 생성!
    model = DynamicReturnNet(X.shape[1], best_h1, best_h2, best_drop)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=best_lr)
    
    # 실전 K-Fold 학습 (충분히 30 에폭)
    model.train()
    for epoch in range(30):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    # 검증 및 테스트 예측
    model.eval()
    with torch.no_grad():
        val_proba = model(torch.FloatTensor(X_va)).squeeze().numpy()
        oof_pred[val_idx] = val_proba
        test_pred += model(X_test_tensor).squeeze().numpy() / skf.n_splits
        
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# OOF 전체 성능 평가
overall_auc = roc_auc_score(y, oof_pred)
print(f"\n🏆 최종 PyTorch OOF ROC-AUC: {overall_auc:.4f}")



📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...
✅ 길이 확인 - X: 200000, y: 200000


[I 2026-04-09 17:56:37,484] A new study created in memory with name: no-name-2cd6e9d7-bcf8-464d-be52-072c5fc8e989



🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 분할 사용)...
🔎 Optuna split 입력 길이 - X_opt_source: 160000, y_opt_source: 160000


[I 2026-04-09 17:57:23,859] Trial 0 finished with value: 0.5884382027751958 and parameters: {'h1': 256, 'h2': 64, 'drop_rate': 0.24912644268490822, 'lr': 0.0033754781887973795}. Best is trial 0 with value: 0.5884382027751958.
[I 2026-04-09 17:57:58,267] Trial 1 finished with value: 0.5885846884098186 and parameters: {'h1': 64, 'h2': 32, 'drop_rate': 0.23316018952821113, 'lr': 0.0017071644234397459}. Best is trial 1 with value: 0.5885846884098186.
[I 2026-04-09 17:58:29,189] Trial 2 finished with value: 0.5889405270751575 and parameters: {'h1': 64, 'h2': 64, 'drop_rate': 0.3633087307413002, 'lr': 0.007227299784297751}. Best is trial 2 with value: 0.5889405270751575.
[I 2026-04-09 17:59:04,083] Trial 3 finished with value: 0.5897092932102496 and parameters: {'h1': 256, 'h2': 32, 'drop_rate': 0.24942573323154754, 'lr': 0.004305448071139076}. Best is trial 3 with value: 0.5897092932102496.
[I 2026-04-09 17:59:32,976] Trial 4 finished with value: 0.5905167804134104 and parameters: {'h1': 64


🎉 탐색 완료! 최적의 파라미터: {'h1': 128, 'h2': 64, 'drop_rate': 0.4157701605943236, 'lr': 0.001961386617942612}

🚀 Optuna 최적 파라미터로 PyTorch K-Fold 학습을 시작합니다...
   - Fold 1 완료 (ROC-AUC: 0.5927)
   - Fold 2 완료 (ROC-AUC: 0.5894)
   - Fold 3 완료 (ROC-AUC: 0.5939)
   - Fold 4 완료 (ROC-AUC: 0.5845)
   - Fold 5 완료 (ROC-AUC: 0.5905)

🏆 최종 PyTorch OOF ROC-AUC: 0.5899


In [6]:
# ==========================================
# 4. 대시보드 연동용 CSV 일괄 추출 (PyTorch Optuna 튜닝모델용)
# ==========================================
import numpy as np
import pandas as pd

model_name = "pytorch_optuna"
print(f"\n💾 {model_name} 대시보드 및 제출용 CSV 파일 추출을 시작합니다...")

# Validation 파트 길이만큼 마지막 구간을 잘라서 저장
val_size = len(X_val_part)

val_prob = np.asarray(oof_pred)[-val_size:]
val_label = (val_prob >= 0.5).astype(int)
val_target = y_val_part.values

# 1. Validation 예측 결과
val_df = pd.DataFrame({
    "pred_prob": val_prob,
    "pred_label": val_label,
    "returned": val_target
})
val_df.to_csv(f"val_predictions_{model_name}.csv", index=False)

# 대시보드에서 일반 DL 모델로 바로 붙일 수 있도록 별칭 파일도 함께 저장
val_df.to_csv("val_predictions_dl.csv", index=False)

# 2. Test 예측 결과 및 제출용
test_prob = np.asarray(test_pred)
test_label = (test_prob >= 0.5).astype(int)
test_df = pd.DataFrame({
    "order_id": test_id_df["order_id"],
    "pred_prob": test_prob,
    "pred_label": test_label
})
test_df.to_csv(f"test_predictions_{model_name}.csv", index=False)
test_df.to_csv("test_predictions_dl.csv", index=False)

test_df[["order_id", "pred_prob"]].rename(columns={"pred_prob": "returned"}).to_csv(
    f"{model_name}_submission.csv", index=False
)

print(f"✅ val_predictions_{model_name}.csv 생성 완료! (rows={len(val_df)})")
print("✅ val_predictions_dl.csv 생성 완료! (Deep Learning 대시보드용)")
print(f"✅ test_predictions_{model_name}.csv 생성 완료! (rows={len(test_df)})")
print("✅ test_predictions_dl.csv 생성 완료! (Deep Learning 대시보드용)")
print(f"✅ {model_name}_submission.csv 생성 완료!")
print(f"✅ 검증 행 수 확인: X_val_part={len(X_val_part)}, y_val_part={len(y_val_part)}, 저장된 val_df={len(val_df)}")
print("⚠️ PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 feature_importance 파일은 생성하지 않습니다.")

print("\n🎉 PyTorch Optuna 튜닝모델 추출 완료!")


💾 pytorch_optuna 대시보드 및 제출용 CSV 파일 추출을 시작합니다...
✅ val_predictions_pytorch_optuna.csv 생성 완료! (rows=40000)
✅ val_predictions_dl.csv 생성 완료! (Deep Learning 대시보드용)
✅ test_predictions_pytorch_optuna.csv 생성 완료! (rows=50000)
✅ test_predictions_dl.csv 생성 완료! (Deep Learning 대시보드용)
✅ pytorch_optuna_submission.csv 생성 완료!
✅ 검증 행 수 확인: X_val_part=40000, y_val_part=40000, 저장된 val_df=40000
⚠️ PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 feature_importance 파일은 생성하지 않습니다.

🎉 PyTorch Optuna 튜닝모델 추출 완료!
